In [ ]:
#!/usr/bin/env python3
# pip install evaluate sacrebleu rouge-score nltk
# pip install rouge-score sacrebleu nltk evaluate
import os
import time
import asyncio
import aiohttp
import pandas as pd
import nltk
import evaluate
import nest_asyncio
from google.colab import drive
from rouge_score import rouge_scorer as rs_lib
import sacrebleu
from nltk.translate.meteor_score import meteor_score as meteor_fn
from nltk.tokenize import word_tokenize
import nltk
from collections import Counter
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nest_asyncio.apply()
drive.mount('/content/drive')

API_KEY         = ""
OPENROUTER_URL  = "https://openrouter.ai/api/v1/chat/completions"
CSV_PATH        = "Health_paraphrasing_sampled_2000.csv"
MODEL           = "qwen/qwen3.7-plus"
OUTPUT_DIR      = "/content/drive/MyDrive/bangla_paraphrase_benchmark"
NUM_SAMPLES     = None
CONCURRENCY     = 5
BATCH_SIZE      = 15
BATCH_DELAY     = 5
SITE_URL        = "https://colab.research.google.com"
SITE_NAME       = "Bangla Paraphrase Benchmark"
MAX_RETRIES     = 1
BASE_BACKOFF    = 2.0
REQUEST_TIMEOUT = 60

SYSTEM_PROMPT = """You are a Bengali paraphrase generation model.
Given a Bengali sentence, generate a single paraphrased version of it.
Rules:
- Preserve the original meaning exactly.
- Vary the vocabulary and sentence structure.
- Output ONLY the paraphrased Bengali sentence, nothing else.
- Do not include explanations, labels, or any extra text."""


def build_user_message(sentence: str) -> str:
    return f"Paraphrase this Bengali sentence:\n{sentence}"

def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def _rouge_n(pred_tokens, ref_tokens, n):
    pred_ng = _ngrams(pred_tokens, n)
    ref_ng  = _ngrams(ref_tokens,  n)
    overlap = sum((pred_ng & ref_ng).values())
    p = overlap / max(sum(pred_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def _lcs_len(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def _rouge_l(pred_tokens, ref_tokens):
    lcs = _lcs_len(pred_tokens, ref_tokens)
    p = lcs / max(len(pred_tokens), 1)
    r = lcs / max(len(ref_tokens),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f

def build_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
        "HTTP-Referer":  SITE_URL,
        "X-Title":       SITE_NAME,
    }


async def call_model_async(session, model, user_message, headers):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        "max_tokens": 256,
        "reasoning": {
          "effort": "none"
        },
        "temperature": 0.3
    }
    start = time.monotonic()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                OPENROUTER_URL, headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
            ) as resp:
                if resp.status == 429:
                    wait = float(resp.headers.get("Retry-After", BASE_BACKOFF * attempt))
                    await asyncio.sleep(wait)
                    continue
                if resp.status >= 500:
                    await asyncio.sleep(BASE_BACKOFF * attempt)
                    continue
                if resp.status != 200:
                    text = await resp.text()
                    return None, f"http_error:{resp.status}:{text[:200]}", time.monotonic() - start
                data    = await resp.json()
                content = data["choices"][0]["message"]["content"].strip()
                return content, None, time.monotonic() - start
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            await asyncio.sleep(BASE_BACKOFF * attempt)
    return None, "max_retries_exceeded", time.monotonic() - start


def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


async def run_model_async(model, df, headers, out_dir):
    existing  = load_checkpoint(out_dir, model)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    sem     = asyncio.Semaphore(CONCURRENCY)
    indices = list(range(start_idx, len(df)))
    batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    results = []

    async def process_row(session, row_idx):
        async with sem:
            row      = df.iloc[row_idx]
            source   = str(row["source_sentence"])
            reference = str(row["paraphrased_sentence"])

            user_msg = build_user_message(source)
            prediction, err, latency = await call_model_async(session, model, user_msg, headers)

            if prediction is None:
                prediction = ""

            print(f"[{row_idx}] err={err} | src={source[:40]}...")

            return {
                "_idx":        row_idx,
                "source":      source,
                "reference":   reference,
                "prediction":  prediction if not err else "",
                "latency":     latency,
                "error":       err,
            }

    async with aiohttp.ClientSession() as session:
        for b_num, batch in enumerate(batches):
            print(f"\nBatch {b_num + 1}/{len(batches)} — rows {batch[0]}..{batch[-1]}")
            tasks         = [process_row(session, i) for i in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

            all_rows = rows_done + sorted(results, key=lambda r: r["_idx"])
            save_checkpoint(out_dir, model, all_rows)

            if b_num < len(batches) - 1:
                print(f"Waiting {BATCH_DELAY}s…")
                await asyncio.sleep(BATCH_DELAY)

    results = sorted(results, key=lambda r: r["_idx"])
    for r in results:
        del r["_idx"]
    all_rows = rows_done + results
    save_checkpoint(out_dir, model, all_rows)
    return all_rows


def run_model_on_dataset(model, df, headers, out_dir):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_model_async(model, df, headers, out_dir))


def compute_metrics(rows):
    predictions, references, latencies = [], [], []

    for r in rows:
        pred = str(r.get("prediction", "")).strip()
        ref  = str(r.get("reference",  "")).strip()
        err  = r.get("error")
        if pred and ref and (err is None or str(err).strip() == ""):
            predictions.append(pred)
            references.append(ref)
        if r.get("latency") is not None:
            latencies.append(float(r["latency"]))

    if not predictions:
        print("No valid predictions found.")
        return {}

    print(f"Evaluating {len(predictions)} valid predictions...")

    # BLEU — char-level for Bengali
    bleu_score = sacrebleu.corpus_bleu(
        predictions, [references], tokenize="none"
    ).score / 100

    # ROUGE — whitespace tokenized
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        pt = pred.split()
        rt = ref.split()
        r1.append(_rouge_n(pt, rt, 1))
        r2.append(_rouge_n(pt, rt, 2))
        rl.append(_rouge_l(pt, rt))

    # METEOR — whitespace tokenized
    meteor_scores = [
        meteor_fn([ref.split()], pred.split())
        for pred, ref in zip(predictions, references)
    ]

    return {
        "BLEU":           bleu_score,
        "ROUGE-1 (F1)":   sum(r1) / len(r1),
        "ROUGE-2 (F1)":   sum(r2) / len(r2),
        "ROUGE-L (F1)":   sum(rl) / len(rl),
        "METEOR":         sum(meteor_scores) / len(meteor_scores),
        "total":          len(predictions),
        "avg_latency":    sum(latencies) / len(latencies) if latencies else None,
    }


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    if not {"source_sentence", "paraphrased_sentence"}.issubset(df.columns):
        raise ValueError("CSV must contain 'source_sentence' and 'paraphrased_sentence' columns")

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"\nDataset: {len(df)} sentence pairs")

    headers = build_headers()
    rows    = run_model_on_dataset(MODEL, df, headers, OUTPUT_DIR)
    metrics = compute_metrics(rows)

    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame([{
        "model":          MODEL,
        "BLEU":           round(metrics["BLEU"], 4),
        "ROUGE-1 (F1)":   round(metrics["ROUGE-1 (F1)"], 4),
        "ROUGE-2 (F1)":   round(metrics["ROUGE-2 (F1)"], 4),
        "ROUGE-L (F1)":   round(metrics["ROUGE-L (F1)"], 4),
        "METEOR":         round(metrics["METEOR"], 4),
        "total_sentences": metrics["total"],
        "avg_latency_s":  metrics["avg_latency"],
    }])
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Evaluation Results (paper Table 5 format) ──")
    paper_scores = {
        "BLEU": 0.1919, "ROUGE-1 (F1)": 0.5160,
        "ROUGE-2 (F1)": 0.2761, "ROUGE-L (F1)": 0.4903, "METEOR": 0.4777
    }
    for metric in ["BLEU", "ROUGE-1 (F1)", "ROUGE-2 (F1)", "ROUGE-L (F1)", "METEOR"]:
        your   = metrics[metric]
        paper  = paper_scores[metric]
        diff   = your - paper
        sign   = "+" if diff >= 0 else ""
        print(f"  {metric:<15} yours={your:.4f}  paper={paper:.4f}  diff={sign}{diff:.4f}")

    print(f"\n  Total evaluated: {metrics['total']} sentences")
    print(f"  Avg latency:     {metrics['avg_latency']:.2f}s" if metrics['avg_latency'] else "")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dataset: 2000 sentence pairs

Batch 1/123 — rows 167..181
[167] err=None | src=তবে এ সময় ভাইরাসটিতে কারও মৃত্যুর তথ্য দ...
[171] err=None | src=তাই ময়দার চেয়ে আটা বেশি স্বাস্থ্যকর...
[168] err=None | src=অন্যান্য উপসর্গগুলো নির্ভর করে আপনার হাড়...
[172] err=None | src=বেশ কিছুদিন লাগামহীন খাওয়াদাওয়া চালিয়ে গ...
[170] err=None | src=সাধারণত অ্যান্টিবায়োটিক গ্রহণ বন্ধ করে ...
[175] err=None | src=পরামর্শ ডেঙ্গু সেরে যাওয়ার পর অনেকের মধ্...
[169] err=None | src=সাধারণত গলাব্যথার সঙ্গে জ্বর থাকে যা অনে...
[177] err=None | src=তবে রাতের খাবারে অবশ্যই লাল আটা অথবা লাল...
[178] err=None | src=গতকাল হায়দরাবাদে ভারত বায়োটেক কারখানায় স...
[173] err=None | src=শনিবার তিনি যুগান্তরকে বলেন করোনা চিকিৎস...
[176] err=None | src= গ্যাস্ট্রাইটিস কিংবা গ্যাস্ট্রিক আলসারে...
[179] err=None | src=বিশ্বে হাজার হাজার মানুষ এই রোগে আক্রান্...
[174] err=None | src=৪ ডায়াবেটিস আছে 